# LeetCode 1236 — Web Crawler bằng Python

Link đề: https://leetcode.com/problems/web-crawler/ · Độ khó: **Medium** · Chủ đề: String, Depth-First Search, Breadth-First Search, Interactive

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề và chữ ký hàm trong notebook được đối chiếu với bản mô tả + lời giải Python tại https://leetcode.doocs.org/en/lc/1236/ (tên method `crawl(self, startUrl, htmlParser)`, interface `HtmlParser.getUrls(url)`).

Skeleton LeetCode:

```python
# """
# This is HtmlParser's API interface.
# You should not implement it, or speculate about its implementation
# """
# class HtmlParser(object):
#    def getUrls(self, url):
#        """
#        :type url: str
#        :rtype list[str]
#        """

class Solution:
    def crawl(self, startUrl: str, htmlParser: 'HtmlParser') -> List[str]:
        ...
```

Ghi chú: đây là bài **Interactive** (tương tác): ta không được nhìn thấy toàn bộ đồ thị trang web, chỉ được "hỏi" `htmlParser.getUrls(url)` để biết trang `url` chứa những link nào. Đáp án trả về **theo thứ tự bất kỳ**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Nhìn ra rằng "các trang web và đường link" thực chất là một **đồ thị có hướng** (directed graph): mỗi URL là một đỉnh, mỗi link là một cạnh.
2. Biết tách **hostname** (tên miền, ví dụ `news.yahoo.com`) ra khỏi một URL dạng `http://hostname/path` một cách an toàn, và tránh bẫy so sánh bằng tiền tố (`startswith`).
3. Cài đặt duyệt đồ thị bằng **DFS** (đệ quy và dùng stack) và **BFS** (dùng `deque`) với tập `visited` để **không crawl một link hai lần**.
4. Hiểu vì sao phải **lọc theo hostname trước khi đi tiếp** (không được đi xuyên qua trang của host khác) và đếm được số lần gọi `getUrls`.
5. Tự viết được một `HtmlParser` giả lập (mock) để kiểm thử bài interactive trên máy của mình.

## 2. Đề bài & ví dụ

Cho một URL `startUrl` và một interface `HtmlParser`. Hãy cài đặt một **web crawler** (trình thu thập trang web) để thu thập **tất cả các link có cùng hostname với `startUrl`**. Trả về mọi URL thu được, theo thứ tự bất kỳ.

Crawler cần:

- Bắt đầu từ trang `startUrl`.
- Gọi `HtmlParser.getUrls(url)` để lấy mọi URL có trên trang `url`.
- **Không crawl cùng một link hai lần.**
- **Chỉ khám phá** những link có **cùng hostname** với `startUrl`.

Giả sử mọi URL dùng giao thức `http` và không có cổng (port). Ví dụ `http://leetcode.com/problems` và `http://leetcode.com/contest` cùng hostname `leetcode.com`; còn `http://example.org/test` và `http://example.com/abc` thì **khác** hostname.

**Lưu ý:** URL có dấu `/` ở cuối được coi là **khác** URL không có: `"http://news.yahoo.com"` và `"http://news.yahoo.com/"` là hai URL khác nhau.

Khi tự test, đề cho ba biến `urls`, `edges`, `startUrl`; `edges[i] = [a, b]` nghĩa là trang `urls[a]` có link tới `urls[b]`. Code của bạn **chỉ** thấy `startUrl` và `htmlParser`.

**Ví dụ 1:** `startUrl = "http://news.yahoo.com/news/topics/"`, kết quả gồm `"http://news.yahoo.com"`, `"http://news.yahoo.com/news"`, `"http://news.yahoo.com/news/topics/"`, `"http://news.yahoo.com/us"`.

**Ví dụ 2:** `startUrl = "http://news.google.com"` → `["http://news.google.com"]` (trang này chỉ link sang các trang khác hostname).

**Ràng buộc:** `1 <= urls.length <= 1000`; `1 <= urls[i].length <= 300`; `startUrl` là một trong `urls`; nhãn hostname dài 1–63 ký tự, chỉ gồm `a-z`, `0-9`, `-`, không bắt đầu/kết thúc bằng `-`.

Vì LeetCode không đưa cho ta `HtmlParser` thật, ta tự dựng một **mock** (đối tượng giả lập) từ `urls` và `edges`. Mock còn **đếm số lần gọi** `getUrls` để ta kiểm tra điều kiện "không crawl hai lần".

In [1]:
import random
import time
from collections import deque, defaultdict
from typing import List


class HtmlParser:
    """Mock HtmlParser: dựng từ urls + edges giống phần custom test của LeetCode."""

    def __init__(self, urls, edges):
        self.links = defaultdict(list)
        for a, b in edges:
            self.links[urls[a]].append(urls[b])
        self.calls = defaultdict(int)   # đếm số lần getUrls(url) được gọi

    def getUrls(self, url):
        self.calls[url] += 1
        return list(self.links[url])


urls1 = [
    "http://news.yahoo.com",
    "http://news.yahoo.com/news",
    "http://news.yahoo.com/news/topics/",
    "http://news.google.com",
    "http://news.yahoo.com/us",
]
edges1 = [[2, 0], [2, 1], [3, 2], [3, 1], [0, 4]]
start1 = "http://news.yahoo.com/news/topics/"
want1 = ["http://news.yahoo.com", "http://news.yahoo.com/news",
         "http://news.yahoo.com/news/topics/", "http://news.yahoo.com/us"]

urls2 = urls1[:4]
edges2 = [[0, 2], [2, 1], [3, 2], [3, 1], [3, 0]]
start2 = "http://news.google.com"
want2 = ["http://news.google.com"]

examples = [(urls1, edges1, start1, want1), (urls2, edges2, start2, want2)]
p = HtmlParser(urls1, edges1)
print(p.getUrls(start1))
print(dict(p.calls))

['http://news.yahoo.com', 'http://news.yahoo.com/news']
{'http://news.yahoo.com/news/topics/': 1}


## 3. Kiến thức nền

### 3.1 Cấu trúc một URL `http://hostname/path`

Một URL trong bài luôn có dạng `http://` + `hostname` + (tuỳ chọn) `/path...`. Hostname là phần nằm **giữa** `http://` và dấu `/` tiếp theo (hoặc tới hết chuỗi nếu không có `/`).

Cách gọn nhất: `url.split('/')`. Vì `http://a.com/x/y` tách theo `/` cho ra `['http:', '', 'a.com', 'x', 'y']`, phần tử chỉ số `2` chính là hostname.

In [2]:
for u in ["http://news.yahoo.com", "http://news.yahoo.com/", "http://news.yahoo.com/news/topics/"]:
    print(repr(u), "->", u.split('/'), "-> host =", u.split('/')[2])

'http://news.yahoo.com' -> ['http:', '', 'news.yahoo.com'] -> host = news.yahoo.com
'http://news.yahoo.com/' -> ['http:', '', 'news.yahoo.com', ''] -> host = news.yahoo.com
'http://news.yahoo.com/news/topics/' -> ['http:', '', 'news.yahoo.com', 'news', 'topics', ''] -> host = news.yahoo.com


### 3.2 Ba cách lấy hostname

- `url.split('/')[2]` — ngắn, dựa vào việc luôn có `http://`.
- `url[7:].split('/')[0]` — bỏ 7 ký tự `"http://"` rồi lấy phần trước `/` đầu tiên (cách của lời giải doocs).
- `urllib.parse.urlparse(url).netloc` — dùng thư viện chuẩn, tổng quát nhất (cũng xử lý được `https`, port...).

Cả ba cho cùng kết quả trên dữ liệu của bài.

In [3]:
from urllib.parse import urlparse


def host_split(url):
    return url.split('/')[2]


def host_slice(url):
    return url[7:].split('/')[0]


def host_parse(url):
    return urlparse(url).netloc


for u in urls1:
    print(f"{u:40} {host_split(u):18} {host_slice(u):18} {host_parse(u)}")
    assert host_split(u) == host_slice(u) == host_parse(u)

http://news.yahoo.com                    news.yahoo.com     news.yahoo.com     news.yahoo.com
http://news.yahoo.com/news               news.yahoo.com     news.yahoo.com     news.yahoo.com
http://news.yahoo.com/news/topics/       news.yahoo.com     news.yahoo.com     news.yahoo.com
http://news.google.com                   news.google.com    news.google.com    news.google.com
http://news.yahoo.com/us                 news.yahoo.com     news.yahoo.com     news.yahoo.com


### 3.3 Tập hợp `set` để đánh dấu đã thăm

`set` trong Python cho phép kiểm tra `x in s` và thêm `s.add(x)` trung bình **O(1)** (dựa trên bảng băm — hash table). Đây là công cụ chuẩn để thực hiện yêu cầu "không crawl cùng link hai lần".

In [4]:
seen = set()
for u in ["a", "b", "a", "c", "b"]:
    if u in seen:
        print(u, "đã thăm -> bỏ qua")
        continue
    seen.add(u)
    print(u, "lần đầu -> thăm")
print(seen == {"a", "b", "c"})

a lần đầu -> thăm
b lần đầu -> thăm
a đã thăm -> bỏ qua
c lần đầu -> thăm
b đã thăm -> bỏ qua
True


### 3.4 `deque` cho BFS và `list` làm stack cho DFS

- **BFS** (Breadth-First Search — duyệt theo chiều rộng): dùng hàng đợi FIFO. `collections.deque` có `append` và `popleft` đều O(1).
- **DFS** (Depth-First Search — duyệt theo chiều sâu): dùng ngăn xếp LIFO. `list` với `append` / `pop()` là đủ, hoặc dùng đệ quy (stack ngầm của Python).

In [5]:
q = deque([1, 2, 3])
q.append(4)
print("popleft:", q.popleft(), "còn", list(q))
st = [1, 2, 3]
st.append(4)
print("pop:", st.pop(), "còn", st)

popleft: 1 còn [2, 3, 4]
pop: 4 còn [1, 2, 3]


### 3.5 Đồ thị có hướng từ danh sách cạnh

`edges = [[2,0],[2,1],[3,2],[3,1],[0,4]]` nghĩa là: trang 2 link tới 0 và 1; trang 3 link tới 2 và 1; trang 0 link tới 4. Link là **một chiều**: trang 0 **không** tự động link về trang 2. Ta in danh sách kề (adjacency list) để nhìn rõ.

In [6]:
adj = defaultdict(list)
for a, b in edges1:
    adj[a].append(b)
for a in sorted(adj):
    print(a, urls1[a], "->", [urls1[b] for b in adj[a]])

0 http://news.yahoo.com -> ['http://news.yahoo.com/us']
2 http://news.yahoo.com/news/topics/ -> ['http://news.yahoo.com', 'http://news.yahoo.com/news']
3 http://news.google.com -> ['http://news.yahoo.com/news/topics/', 'http://news.yahoo.com/news']


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1. Host cần giữ: `host(startUrl) = news.yahoo.com`.

| Bước | Đang thăm | `getUrls` trả về | Link cùng host chưa thăm → đưa vào |
|---|---|---|---|
| 1 | `.../news/topics/` (2) | 0, 1 | 0, 1 |
| 2 | `news.yahoo.com` (0) | 4 | 4 |
| 3 | `.../news` (1) | (không có) | — |
| 4 | `.../us` (4) | (không có) | — |

Trang 3 (`news.google.com`) không bao giờ được chạm tới: không có trang nào cùng host link tới nó, và dù có thì nó **khác host** nên cũng bị bỏ.

Kết quả: {0, 1, 2, 4} — đúng như đề. Đây chính là: **tập các đỉnh đến được từ `startUrl` nếu chỉ được đi qua các đỉnh cùng host**.

In [7]:
# Kiểm tra trực giác bằng cách "đi tay" đúng như bảng trên
target = host_split(start1)
visited = {start1}
order = [start1]
i = 0
while i < len(order):
    cur = order[i]
    i += 1
    for nxt in HtmlParser(urls1, edges1).getUrls(cur):
        if host_split(nxt) == target and nxt not in visited:
            visited.add(nxt)
            order.append(nxt)
print(order)
assert sorted(order) == sorted(want1)

['http://news.yahoo.com/news/topics/', 'http://news.yahoo.com', 'http://news.yahoo.com/news', 'http://news.yahoo.com/us']


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng "lặp tới điểm bất động" (fixpoint):** giữ tập kết quả `res = {startUrl}`. Lặp lại: với **mọi** URL trong `res`, gọi `getUrls`, thêm các link cùng host vào `res`. Dừng khi một vòng không thêm được URL mới.

Cách này đúng nhưng **gọi lại `getUrls` cho cùng một URL rất nhiều lần** (mỗi vòng gọi lại toàn bộ), vi phạm tinh thần "không crawl cùng link hai lần" và tốn thời gian.

In [8]:
class SolutionFixpoint:
    def crawl(self, startUrl, htmlParser):
        host = startUrl.split('/')[2]
        res = {startUrl}
        changed = True
        while changed:
            changed = False
            for u in list(res):                  # crawl lại TẤT CẢ mỗi vòng
                for v in htmlParser.getUrls(u):
                    if v.split('/')[2] == host and v not in res:
                        res.add(v)
                        changed = True
        return list(res)


for urls, edges, start, want in examples:
    p = HtmlParser(urls, edges)
    got = SolutionFixpoint().crawl(start, p)
    print(sorted(got) == sorted(want), "| tổng số lần gọi getUrls:", sum(p.calls.values()))

True | tổng số lần gọi getUrls: 8
True | tổng số lần gọi getUrls: 1


**Độ phức tạp:** gọi `R` là số URL kết quả, `E` là số link trên các trang đó. Mỗi vòng tốn `O(E)` (cộng chi phí xử lý chuỗi), và trong trường hợp xấu (chuỗi dài `a → b → c → ...`) cần `R` vòng → **O(R · E)** lần xử lý link và **O(R²)** lần gọi `getUrls`.

**Nút thắt:** ta quên mất ai **đã được crawl rồi**. Chỉ cần ghi nhớ điều này, mỗi URL được crawl đúng một lần.

In [9]:
# Minh hoạ nút thắt: chuỗi dài a0 -> a1 -> ... -> a(k-1)
k = 30
chain_urls = [f"http://a.com/{i}" for i in range(k)]
chain_edges = [[i, i + 1] for i in range(k - 1)]
p = HtmlParser(chain_urls, chain_edges)
SolutionFixpoint().crawl(chain_urls[0], p)
print("k =", k, "-> số lần gọi getUrls của fixpoint:", sum(p.calls.values()))

k = 30 -> số lần gọi getUrls của fixpoint: 465


## 6. Bước đột phá

**Insight:** mô hình hoá thành bài **duyệt đồ thị** quen thuộc:

- Đỉnh = URL; cạnh `u → v` nếu trang `u` chứa link `v`.
- Ta chỉ được **đi vào** đỉnh `v` khi `host(v) == host(startUrl)`.
- Mỗi đỉnh chỉ **thăm (crawl) một lần** nhờ tập `visited`.

**Invariant (bất biến) của BFS/DFS:** mọi URL nằm trong `visited` đều (1) cùng host với `startUrl`, (2) đến được từ `startUrl` qua các trang cùng host, và (3) đã hoặc sẽ được crawl đúng một lần.

**Quy tắc "nếu… thì…":**

- Nếu `v` khác host → bỏ qua ngay, **không** crawl `v` (vì đề cấm khám phá trang khác host).
- Nếu `v` cùng host nhưng đã ở `visited` → bỏ qua.
- Nếu `v` cùng host và chưa thăm → thêm vào `visited` **ngay lúc phát hiện** và đưa vào hàng đợi/stack.

Một chi tiết hay: chỉ cần so sánh host với `host(startUrl)` (tính **một lần**), vì mọi URL ta crawl đều cùng host này.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy (giống lời giải doocs)

Hàm `dfs(url)`: nếu `url` đã thăm thì dừng; ngược lại đánh dấu, rồi gọi `dfs` cho mọi link cùng host.

In [10]:
class SolutionDFSRecursive:
    def crawl(self, startUrl, htmlParser):
        host = startUrl.split('/')[2]
        ans = set()

        def dfs(url):
            if url in ans:
                return
            ans.add(url)
            for nxt in htmlParser.getUrls(url):
                if nxt.split('/')[2] == host:
                    dfs(nxt)

        dfs(startUrl)
        return list(ans)


for urls, edges, start, want in examples:
    p = HtmlParser(urls, edges)
    got = SolutionDFSRecursive().crawl(start, p)
    print(sorted(got) == sorted(want), "| max lần gọi cho 1 url:", max(p.calls.values()))

True | max lần gọi cho 1 url: 1
True | max lần gọi cho 1 url: 1


**Độ phức tạp:** mỗi URL kết quả được crawl đúng một lần → `R` lần gọi `getUrls`; mỗi link được xét một lần → `O(E · L)` với `L` là độ dài URL (cắt chuỗi, băm chuỗi). Bộ nhớ `O(R · L)` cho `ans` + độ sâu đệ quy tối đa `R`.

**Lưu ý đệ quy:** với `urls.length <= 1000`, độ sâu tối đa ~1000 — sát giới hạn mặc định của Python (`sys.getrecursionlimit()` thường là 1000). Trên LeetCode vẫn qua vì giới hạn ở đó được nới, nhưng an toàn hơn là dùng bản lặp (Cách 3, 4).

In [11]:
import sys
print("recursion limit mặc định:", sys.getrecursionlimit())

recursion limit mặc định: 1000


### 7.2 Cách 3 — BFS với `deque`

BFS crawl theo "từng lớp": trước hết các trang cách `startUrl` 1 link, rồi 2 link... Đánh dấu `visited` **ngay khi đưa vào hàng đợi** để một URL không bị đưa vào hai lần.

In [12]:
class SolutionBFS:
    def crawl(self, startUrl, htmlParser):
        host = startUrl.split('/')[2]
        visited = {startUrl}
        q = deque([startUrl])
        while q:
            url = q.popleft()
            for nxt in htmlParser.getUrls(url):
                if nxt not in visited and nxt.split('/')[2] == host:
                    visited.add(nxt)
                    q.append(nxt)
        return list(visited)


for urls, edges, start, want in examples:
    p = HtmlParser(urls, edges)
    got = SolutionBFS().crawl(start, p)
    print(sorted(got) == sorted(want), dict(p.calls))

True {'http://news.yahoo.com/news/topics/': 1, 'http://news.yahoo.com': 1, 'http://news.yahoo.com/news': 1, 'http://news.yahoo.com/us': 1}
True {'http://news.google.com': 1}


**Độ phức tạp:** như Cách 2: `R` lần gọi `getUrls`, `O(E · L)` xử lý chuỗi, bộ nhớ `O(R · L)`. Không có rủi ro tràn stack đệ quy.

Mẹo nhỏ: viết `nxt not in visited` **trước** `split` giúp bỏ qua sớm các URL đã thăm mà không cần cắt chuỗi.

### 7.3 Cách 4 — DFS lặp với stack tường minh

Giống BFS, chỉ đổi `deque.popleft()` thành `list.pop()`. Thứ tự crawl khác nhưng **tập kết quả giống hệt**.

In [13]:
class SolutionDFSStack:
    def crawl(self, startUrl, htmlParser):
        host = startUrl[7:].split('/')[0]
        visited = {startUrl}
        stack = [startUrl]
        while stack:
            url = stack.pop()
            for nxt in htmlParser.getUrls(url):
                if nxt not in visited and nxt[7:].split('/')[0] == host:
                    visited.add(nxt)
                    stack.append(nxt)
        return list(visited)


for urls, edges, start, want in examples:
    p = HtmlParser(urls, edges)
    print(sorted(SolutionDFSStack().crawl(start, p)) == sorted(want))

True
True


### 7.4 Cách 5 — BFS dùng `urlparse` + trả về theo thứ tự phát hiện

Biến thể "chuyên nghiệp" hơn: dùng `urllib.parse.urlparse` để lấy `netloc` (phần host[:port]). Ta trả về một `list` giữ thứ tự phát hiện (thứ tự BFS) — tiện cho debug. Đề chấp nhận mọi thứ tự nên đây chỉ là sở thích.

In [14]:
class SolutionBFSParse:
    def crawl(self, startUrl, htmlParser):
        host = urlparse(startUrl).netloc
        order = [startUrl]
        visited = {startUrl}
        i = 0
        while i < len(order):          # dùng chính list làm hàng đợi, i là "đầu hàng"
            url = order[i]
            i += 1
            for nxt in htmlParser.getUrls(url):
                if nxt not in visited and urlparse(nxt).netloc == host:
                    visited.add(nxt)
                    order.append(nxt)
        return order


for urls, edges, start, want in examples:
    print(SolutionBFSParse().crawl(start, HtmlParser(urls, edges)))

['http://news.yahoo.com/news/topics/', 'http://news.yahoo.com', 'http://news.yahoo.com/news', 'http://news.yahoo.com/us']
['http://news.google.com']


`urlparse` chậm hơn `split` một chút (nó phân tích cả scheme, path, query, fragment) nhưng an toàn khi URL có dạng tổng quát (`https`, `user@host:port`...). Trong bài này cả hai đều đúng.

## 8. TRACE từng bước

Trace BFS trên ví dụ 1 (đánh số URL theo chỉ số trong `urls1`):

| Vòng | Lấy ra | `getUrls` | Thêm vào hàng đợi | Hàng đợi sau vòng | `visited` |
|---|---|---|---|---|---|
| 0 | — | — | 2 | [2] | {2} |
| 1 | 2 | [0, 1] | 0, 1 | [0, 1] | {2, 0, 1} |
| 2 | 0 | [4] | 4 | [1, 4] | {2, 0, 1, 4} |
| 3 | 1 | [] | — | [4] | {2, 0, 1, 4} |
| 4 | 4 | [] | — | [] | {2, 0, 1, 4} |

Hàm `trace_bfs` in đúng các cột trên.

In [15]:
def trace_bfs(urls, edges, startUrl):
    idx = {u: i for i, u in enumerate(urls)}
    parser = HtmlParser(urls, edges)
    host = startUrl.split('/')[2]
    visited = {startUrl}
    q = deque([startUrl])
    step = 0
    print(f"vòng 0: queue={[idx[u] for u in q]} visited={sorted(idx[u] for u in visited)}")
    while q:
        step += 1
        url = q.popleft()
        got = parser.getUrls(url)
        added, skipped = [], []
        for nxt in got:
            if nxt in visited:
                skipped.append((idx[nxt], "đã thăm"))
            elif nxt.split('/')[2] != host:
                skipped.append((idx[nxt], "khác host"))
            else:
                visited.add(nxt)
                q.append(nxt)
                added.append(idx[nxt])
        print(f"vòng {step}: lấy {idx[url]} | getUrls={[idx[u] for u in got]} | thêm={added} "
              f"| bỏ={skipped} | queue={[idx[u] for u in q]}")
    return sorted(visited)


res = trace_bfs(urls1, edges1, start1)
print(res)

vòng 0: queue=[2] visited=[2]
vòng 1: lấy 2 | getUrls=[0, 1] | thêm=[0, 1] | bỏ=[] | queue=[0, 1]
vòng 2: lấy 0 | getUrls=[4] | thêm=[4] | bỏ=[] | queue=[1, 4]
vòng 3: lấy 1 | getUrls=[] | thêm=[] | bỏ=[] | queue=[4]
vòng 4: lấy 4 | getUrls=[] | thêm=[] | bỏ=[] | queue=[]
['http://news.yahoo.com', 'http://news.yahoo.com/news', 'http://news.yahoo.com/news/topics/', 'http://news.yahoo.com/us']


Trace ví dụ 2: bắt đầu ở `news.google.com` (chỉ số 3); mọi link đều sang `news.yahoo.com` nên bị bỏ vì **khác host**.

In [16]:
res = trace_bfs(urls2, edges2, start2)
print(res)

vòng 0: queue=[3] visited=[3]
vòng 1: lấy 3 | getUrls=[2, 1, 0] | thêm=[] | bỏ=[(2, 'khác host'), (1, 'khác host'), (0, 'khác host')] | queue=[]
['http://news.google.com']


## 9. Code nộp LeetCode

Chọn BFS lặp (không lo giới hạn đệ quy, dễ đọc).

In [17]:
# """
# This is HtmlParser's API interface.
# You should not implement it, or speculate about its implementation
# """
# class HtmlParser(object):
#    def getUrls(self, url):
#        """
#        :type url: str
#        :rtype list[str]
#        """

from collections import deque
from typing import List


class Solution:
    def crawl(self, startUrl: str, htmlParser: 'HtmlParser') -> List[str]:
        host = startUrl.split('/')[2]
        visited = {startUrl}
        q = deque([startUrl])
        while q:
            url = q.popleft()
            for nxt in htmlParser.getUrls(url):
                if nxt not in visited and nxt.split('/')[2] == host:
                    visited.add(nxt)
                    q.append(nxt)
        return list(visited)


for urls, edges, start, want in examples:
    p = HtmlParser(urls, edges)
    got = Solution().crawl(start, p)
    assert sorted(got) == sorted(want)
    assert all(c == 1 for c in p.calls.values())   # mỗi url crawl đúng 1 lần
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao đúng

Gọi `H = host(startUrl)` và `G_H` là đồ thị chỉ gồm các URL có host `H`. Đáp án đúng là tập `Reach` = các đỉnh đến được từ `startUrl` **trong `G_H`**.

1. **Không thừa (soundness):** một URL chỉ vào `visited` khi nó cùng host `H` và được tìm thấy trên trang của một URL đã ở `visited`. Quy nạp theo thứ tự thêm vào: mọi URL trong `visited` đến được từ `startUrl` qua chuỗi URL cùng host → thuộc `Reach`.
2. **Không thiếu (completeness):** giả sử `v ∈ Reach` nhưng không trong `visited`. Lấy đường `startUrl = u0 → u1 → … → uk = v` trong `G_H`, và `ui` là đỉnh **đầu tiên** không trong `visited` (`i ≥ 1`). Khi đó `u(i-1)` trong `visited`, nên nó đã được lấy ra khỏi hàng đợi và crawl; lúc đó link `ui` được xét, cùng host, chưa thăm → bị thêm vào. Mâu thuẫn.
3. **Không crawl hai lần:** URL được thêm vào hàng đợi đúng một lần (lúc thêm vào `visited`), và chỉ URL lấy ra khỏi hàng đợi mới được gọi `getUrls`.
4. **Dừng:** mỗi vòng lấy ra một URL, tổng số URL hữu hạn (≤ 1000).

Vì sao **không được** đi qua trang khác host: đề yêu cầu "chỉ khám phá link cùng host". Nếu crawl trang khác host rồi lấy link cùng host trên đó, ta sẽ trả về những URL mà đáp án chuẩn không có (xem Lỗi 2 ở mục 13).

## 11. Time / Space Complexity

Đặt:

- `R` = số URL trong kết quả (≤ `N = urls.length ≤ 1000`),
- `E` = tổng số link trên các trang được crawl,
- `L` = độ dài URL tối đa (≤ 300).

| Cách | Số lần gọi `getUrls` | Time (xử lý chuỗi) | Space |
|---|---|---|---|
| Fixpoint (Cách 1) | `O(R²)` worst | `O(R · E · L)` | `O(R · L)` |
| DFS đệ quy | `R` | `O(E · L)` | `O(R · L)` + stack đệ quy `O(R)` |
| BFS / DFS lặp | `R` | `O(E · L)` | `O(R · L)` |

Chi phí `L` đến từ việc cắt chuỗi lấy host và băm chuỗi khi tra `set`. Trong thực tế, chi phí thật của crawler nằm ở **số lần tải trang** (`getUrls`), nên tối ưu quan trọng nhất là gọi mỗi URL đúng một lần.

## 12. Bộ test

### 12.1 Ví dụ đề + edge cases

In [18]:
ALL = [SolutionFixpoint, SolutionDFSRecursive, SolutionBFS, SolutionDFSStack, SolutionBFSParse, Solution]


def run_all(urls, edges, start):
    outs = []
    for cls in ALL:
        outs.append(sorted(cls().crawl(start, HtmlParser(urls, edges))))
    assert all(o == outs[0] for o in outs), outs
    return outs[0]


# ví dụ đề
for urls, edges, start, want in examples:
    assert run_all(urls, edges, start) == sorted(want)

# edge 1: chỉ có 1 url, không có cạnh
assert run_all(["http://a.com"], [], "http://a.com") == ["http://a.com"]

# edge 2: tự link về chính mình + chu trình
u = ["http://a.com", "http://a.com/x"]
assert run_all(u, [[0, 0], [0, 1], [1, 0]], u[0]) == sorted(u)

# edge 3: dấu "/" cuối là URL khác
u = ["http://a.com", "http://a.com/"]
assert run_all(u, [[0, 1]], u[0]) == sorted(u)
assert run_all(u, [], u[1]) == ["http://a.com/"]

# edge 4: host là tiền tố của host khác (bẫy startswith)
u = ["http://leetcode.com", "http://leetcode.com.cn/x", "http://leetcode.co/y"]
assert run_all(u, [[0, 1], [0, 2]], u[0]) == ["http://leetcode.com"]
print("edge cases OK")

edge cases OK


### 12.2 Phản ví dụ: trang cùng host chỉ đến được **qua** trang khác host

`a.com/1 → b.com → a.com/2`. Đáp án chỉ có `a.com/1` (không được đi qua `b.com`).

In [19]:
u = ["http://a.com/1", "http://b.com", "http://a.com/2"]
print(run_all(u, [[0, 1], [1, 2]], u[0]))
assert run_all(u, [[0, 1], [1, 2]], u[0]) == ["http://a.com/1"]

['http://a.com/1']


### 12.3 Ngẫu nhiên có seed: so mọi cách với tham chiếu tính trực tiếp trên đồ thị

Tham chiếu `reference` biết toàn bộ `urls`/`edges` (thứ mà lời giải không được biết) và tính tập đến được trong đồ thị con cùng host.

In [20]:
def reference(urls, edges, start):
    host = start.split('/')[2]
    adj = defaultdict(list)
    for a, b in edges:
        adj[a].append(b)
    s = urls.index(start)
    seen = {s}
    st = [s]
    while st:
        x = st.pop()
        for y in adj[x]:
            if y not in seen and urls[y].split('/')[2] == host:
                seen.add(y)
                st.append(y)
    return sorted(urls[i] for i in seen)


def random_case(rng, n, m):
    hosts = ["a.com", "a.co", "b.a.com", "a.com.vn"]
    urls = set()
    while len(urls) < n:
        h = rng.choice(hosts)
        path = rng.choice(["", "/", "/" + str(rng.randint(0, 50)), "/p/" + str(rng.randint(0, 9)) + "/"])
        urls.add("http://" + h + path)
    urls = sorted(urls)
    edges = [[rng.randrange(n), rng.randrange(n)] for _ in range(m)]
    return urls, edges, rng.choice(urls)


rng = random.Random(1236)
for _ in range(400):
    n = rng.randint(1, 25)
    urls, edges, start = random_case(rng, n, rng.randint(0, 3 * n))
    want = reference(urls, edges, start)
    assert run_all(urls, edges, start) == want
    p = HtmlParser(urls, edges)
    Solution().crawl(start, p)
    assert all(c == 1 for c in p.calls.values())
print("400 test ngẫu nhiên OK")

400 test ngẫu nhiên OK


### 12.4 Test giới hạn: 1000 URL, nhiều cạnh

In [21]:
rng = random.Random(7)
N = 1000
big_urls = [f"http://news.yahoo.com/{i}" if i % 3 else f"http://news.google.com/{i}" for i in range(N)]
big_edges = [[rng.randrange(N), rng.randrange(N)] for _ in range(20000)]
start = big_urls[1]
p = HtmlParser(big_urls, big_edges)
t = time.perf_counter()
got = Solution().crawl(start, p)
dt = time.perf_counter() - t
assert sorted(got) == reference(big_urls, big_edges, start)
print(len(got), "url | max lần gọi/url =", max(p.calls.values()), f"| {dt * 1000:.1f} ms")

666 url | max lần gọi/url = 1 | 1.3 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh host bằng `startswith`

Code sai: coi `nxt` cùng host nếu `nxt.startswith("http://" + host)`.

In [22]:
def crawl_wrong_prefix(startUrl, htmlParser):
    host = startUrl.split('/')[2]
    visited = {startUrl}
    q = deque([startUrl])
    while q:
        url = q.popleft()
        for nxt in htmlParser.getUrls(url):
            if nxt not in visited and nxt.startswith("http://" + host):   # SAI
                visited.add(nxt)
                q.append(nxt)
    return sorted(visited)


u = ["http://leetcode.com", "http://leetcode.com.cn/x"]
print("sai :", crawl_wrong_prefix(u[0], HtmlParser(u, [[0, 1]])))
print("đúng:", sorted(Solution().crawl(u[0], HtmlParser(u, [[0, 1]]))))

sai : ['http://leetcode.com', 'http://leetcode.com.cn/x']
đúng: ['http://leetcode.com']


**Vì sao sai:** `"http://leetcode.com.cn/x"` bắt đầu bằng `"http://leetcode.com"` nhưng host là `leetcode.com.cn`. **Sửa:** tách host đầy đủ rồi so sánh bằng `==` (như `Solution`).

### Lỗi 2 — Crawl cả trang khác host rồi mới lọc kết quả

Code sai: duyệt **mọi** trang đến được (kể cả khác host), cuối cùng mới lọc theo host.

In [23]:
def crawl_wrong_filter_late(startUrl, htmlParser):
    host = startUrl.split('/')[2]
    visited = {startUrl}
    q = deque([startUrl])
    while q:
        url = q.popleft()
        for nxt in htmlParser.getUrls(url):
            if nxt not in visited:                 # SAI: không chặn khác host
                visited.add(nxt)
                q.append(nxt)
    return sorted(v for v in visited if v.split('/')[2] == host)


u = ["http://a.com/1", "http://b.com", "http://a.com/2"]
print("sai :", crawl_wrong_filter_late(u[0], HtmlParser(u, [[0, 1], [1, 2]])))
print("đúng:", sorted(Solution().crawl(u[0], HtmlParser(u, [[0, 1], [1, 2]]))))

sai : ['http://a.com/1', 'http://a.com/2']
đúng: ['http://a.com/1']


**Vì sao sai:** `a.com/2` chỉ đến được **qua** `b.com`, mà đề cấm khám phá trang khác host → không được có trong đáp án. Thêm nữa, crawl trang khác host là lãng phí (có thể là cả Internet!). **Sửa:** kiểm tra host **trước khi** đưa vào hàng đợi.

### Lỗi 3 — Đánh dấu `visited` lúc lấy ra thay vì lúc đưa vào

Code sai: chỉ thêm vào `visited` khi `popleft`; khi đưa vào hàng đợi không kiểm tra hàng đợi đã có URL đó chưa.

In [24]:
def crawl_wrong_mark_late(startUrl, htmlParser):
    host = startUrl.split('/')[2]
    visited = set()
    q = deque([startUrl])
    while q:
        url = q.popleft()
        visited.add(url)                           # đánh dấu muộn
        for nxt in htmlParser.getUrls(url):
            if nxt not in visited and nxt.split('/')[2] == host:
                q.append(nxt)                      # có thể thêm trùng!
    return sorted(visited)


# kim cương: 0 -> 1, 0 -> 2, 1 -> 3, 2 -> 3 : trang 3 bị crawl 2 lần
u = [f"http://a.com/{i}" for i in range(4)]
e = [[0, 1], [0, 2], [1, 3], [2, 3]]
p_bad = HtmlParser(u, e)
crawl_wrong_mark_late(u[0], p_bad)
p_ok = HtmlParser(u, e)
Solution().crawl(u[0], p_ok)
print("sai : số lần crawl", dict(p_bad.calls))
print("đúng: số lần crawl", dict(p_ok.calls))

sai : số lần crawl {'http://a.com/0': 1, 'http://a.com/1': 1, 'http://a.com/2': 1, 'http://a.com/3': 2}
đúng: số lần crawl {'http://a.com/0': 1, 'http://a.com/1': 1, 'http://a.com/2': 1, 'http://a.com/3': 1}


**Vì sao sai:** tập kết quả vẫn đúng nhưng trang 3 bị crawl **hai lần** — vi phạm yêu cầu "không crawl cùng link hai lần", và trên đồ thị dày, hàng đợi có thể phình rất to. **Sửa:** `visited.add(nxt)` ngay lúc `q.append(nxt)`, và khởi tạo `visited = {startUrl}`.

### Lỗi 4 — "Chuẩn hoá" URL bằng cách bỏ dấu `/` cuối

In [25]:
def crawl_wrong_strip(startUrl, htmlParser):
    host = startUrl.split('/')[2]
    visited = {startUrl.rstrip('/')}
    q = deque([startUrl])
    while q:
        url = q.popleft()
        for nxt in htmlParser.getUrls(url):
            key = nxt.rstrip('/')                  # SAI: gộp "x" và "x/"
            if key not in visited and nxt.split('/')[2] == host:
                visited.add(key)
                q.append(nxt)
    return sorted(visited)


u = ["http://a.com", "http://a.com/"]
print("sai :", crawl_wrong_strip(u[0], HtmlParser(u, [[0, 1]])))
print("đúng:", sorted(Solution().crawl(u[0], HtmlParser(u, [[0, 1]]))))

sai : ['http://a.com']
đúng: ['http://a.com', 'http://a.com/']


**Vì sao sai:** đề nói rõ hai URL này **khác nhau**, nên phải giữ nguyên chuỗi. **Sửa:** dùng chính chuỗi `url` làm khoá trong `visited`.

## 14. Mẹo & nhận dạng pattern

- Thấy "trang web + link", "bạn bè của bạn bè", "phòng có chìa khoá mở phòng khác" → nghĩ ngay **duyệt đồ thị** (BFS/DFS + `visited`).
- Bài **Interactive**: ta không có danh sách cạnh, nhưng có hàm "cho tôi hàng xóm của đỉnh `u`" — vẫn là BFS/DFS y hệt, chỉ thay `adj[u]` bằng `getUrls(u)`.
- Điều kiện lọc đỉnh (cùng host, ô không phải tường, giá trị ≤ ngưỡng...) đặt **trước khi** đẩy vào hàng đợi.
- Đánh dấu `visited` **lúc đẩy vào**, không phải lúc lấy ra.
- Chuỗi có cấu trúc cố định (`http://host/...`) → `split` theo ký tự phân cách, so sánh **cả trường** bằng `==`, đừng so tiền tố.
- Tự viết **mock** cho bài interactive để test offline, và cho mock **đếm số lần gọi**.

## 15. Biến thể

1. **LeetCode 1242 — Web Crawler Multithreaded:** cùng đề nhưng `getUrls` chậm, phải crawl song song bằng nhiều luồng (thread pool), cần khoá (`Lock`) bảo vệ `visited`.
2. **Giới hạn độ sâu:** chỉ crawl các trang cách `startUrl` tối đa `d` link → BFS theo lớp, dừng khi tới lớp `d`.
3. **Cho phép subdomain:** coi `blog.a.com` cùng "site" với `a.com` → so sánh hậu tố theo **nhãn** (`host == H or host.endswith('.' + H)`), không phải hậu tố chuỗi thô.
4. **Trả về theo thứ tự BFS / số bước tới mỗi trang:** lưu thêm `dist[url]`.
5. **LeetCode 841 — Keys and Rooms**, **LeetCode 133 — Clone Graph**: cùng khung "duyệt đồ thị từ một đỉnh, đánh dấu đã thăm".

Demo biến thể 2 (giới hạn độ sâu) và 3 (cho phép subdomain):

In [26]:
def crawl_depth(startUrl, htmlParser, d):
    host = startUrl.split('/')[2]
    visited = {startUrl}
    layer = [startUrl]
    for _ in range(d):
        nxt_layer = []
        for url in layer:
            for nxt in htmlParser.getUrls(url):
                if nxt not in visited and nxt.split('/')[2] == host:
                    visited.add(nxt)
                    nxt_layer.append(nxt)
        layer = nxt_layer
    return sorted(visited)


def same_site(h, base):
    return h == base or h.endswith('.' + base)


def crawl_subdomain(startUrl, htmlParser):
    base = startUrl.split('/')[2]
    visited = {startUrl}
    q = deque([startUrl])
    while q:
        url = q.popleft()
        for nxt in htmlParser.getUrls(url):
            if nxt not in visited and same_site(nxt.split('/')[2], base):
                visited.add(nxt)
                q.append(nxt)
    return sorted(visited)


u = [f"http://a.com/{i}" for i in range(5)]
e = [[i, i + 1] for i in range(4)]
print("độ sâu 2:", crawl_depth(u[0], HtmlParser(u, e), 2))
u2 = ["http://a.com", "http://blog.a.com", "http://xa.com", "http://a.com.vn"]
print("subdomain:", crawl_subdomain(u2[0], HtmlParser(u2, [[0, 1], [0, 2], [0, 3]])))

độ sâu 2: ['http://a.com/0', 'http://a.com/1', 'http://a.com/2']
subdomain: ['http://a.com', 'http://blog.a.com']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Fixpoint | Lặp crawl lại toàn bộ tới khi không đổi | `O(R · E · L)`, `O(R²)` lần gọi | `O(R · L)` | Chỉ để hiểu / làm tham chiếu |
| 2. DFS đệ quy | `dfs(url)` + `set` | `O(E · L)`, `R` lần gọi | `O(R · L)` + stack đệ quy | Code ngắn, đồ thị nhỏ/nông |
| 3. BFS `deque` | Hàng đợi, đánh dấu lúc đẩy vào | `O(E · L)`, `R` lần gọi | `O(R · L)` | Mặc định, an toàn |
| 4. DFS stack | Như BFS nhưng `pop()` | `O(E · L)`, `R` lần gọi | `O(R · L)` | Khi thích DFS mà tránh đệ quy |
| 5. BFS `urlparse` | Như 3, host bằng thư viện chuẩn | `O(E · L)` (hằng số lớn hơn) | `O(R · L)` | URL tổng quát (https, port) |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** với `urls = ["http://x.com", "http://x.com/a", "http://y.com", "http://x.com/b"]`, `edges = [[0,2],[2,3],[0,1]]`, `startUrl = "http://x.com"`, `Solution().crawl` trả về những URL nào?

**Bài 2 (tính tay):** cũng dữ liệu trên nhưng thêm cạnh `[1,3]`. Có bao nhiêu lần gọi `getUrls` tổng cộng?

**Bài 3 (điền TODO):** hoàn thiện `get_host(url)` trả về hostname của URL dạng `http://host/...`.

**Bài 4 (tự viết lại):** viết `crawl_dfs(startUrl, htmlParser)` bằng DFS stack, trả về `list` đã sort.

**Bài 5 (mini challenge):** viết `crawl_with_dist(startUrl, htmlParser)` trả về `dict` `{url: số link ít nhất để đi từ startUrl tới url}` (chỉ đi qua trang cùng host).

In [27]:
def get_host(url):
    # TODO: trả về hostname, ví dụ get_host("http://a.com/x") == "a.com"
    raise NotImplementedError


def crawl_dfs(startUrl, htmlParser):
    # TODO: DFS bằng stack, mỗi url crawl đúng 1 lần, trả về sorted list
    raise NotImplementedError


def crawl_with_dist(startUrl, htmlParser):
    # TODO: BFS, trả về dict url -> khoảng cách
    raise NotImplementedError


# Test gợi ý (bỏ comment sau khi làm xong):
# assert get_host("http://a.com/x/y") == "a.com" and get_host("http://a.com") == "a.com"
# assert crawl_dfs(start1, HtmlParser(urls1, edges1)) == sorted(want1)
# assert crawl_with_dist(start1, HtmlParser(urls1, edges1))["http://news.yahoo.com/us"] == 2

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: chỉ đi theo cạnh tới URL host `x.com`.
- Mức 2: `y.com` bị chặn nên cạnh `2 → 3` không bao giờ được dùng.
- Mức 3: mỗi URL kết quả được gọi `getUrls` đúng một lần → số lần gọi = số URL kết quả.

**Bài 3:**
- Mức 1: URL luôn bắt đầu bằng `http://`.
- Mức 2: `split('/')` cho `['http:', '', host, ...]`.
- Mức 3: `return url.split('/')[2]`.

**Bài 4–5:**
- Mức 1: chép khung BFS ở mục 9.
- Mức 2: Bài 4 đổi `popleft` → `pop`; Bài 5 dùng `dist` thay cho `visited`.
- Mức 3: `dist[nxt] = dist[url] + 1` ngay khi phát hiện `nxt` lần đầu (BFS đảm bảo đó là khoảng cách ngắn nhất).

## 19. Đáp án tham khảo

**Bài 1:** `["http://x.com", "http://x.com/a"]` — `x.com/b` chỉ đến được qua `y.com`.

**Bài 2:** thêm `1 → 3` thì `x.com/b` đến được qua `x.com/a`; kết quả có 3 URL nên có **3** lần gọi `getUrls`.

In [28]:
u = ["http://x.com", "http://x.com/a", "http://y.com", "http://x.com/b"]
e = [[0, 2], [2, 3], [0, 1]]
assert sorted(Solution().crawl(u[0], HtmlParser(u, e))) == ["http://x.com", "http://x.com/a"]
p = HtmlParser(u, e + [[1, 3]])
assert len(Solution().crawl(u[0], p)) == 3 and sum(p.calls.values()) == 3
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [29]:
def get_host(url):
    return url.split('/')[2]


def crawl_dfs(startUrl, htmlParser):
    host = get_host(startUrl)
    visited = {startUrl}
    stack = [startUrl]
    while stack:
        url = stack.pop()
        for nxt in htmlParser.getUrls(url):
            if nxt not in visited and get_host(nxt) == host:
                visited.add(nxt)
                stack.append(nxt)
    return sorted(visited)


def crawl_with_dist(startUrl, htmlParser):
    host = get_host(startUrl)
    dist = {startUrl: 0}
    q = deque([startUrl])
    while q:
        url = q.popleft()
        for nxt in htmlParser.getUrls(url):
            if nxt not in dist and get_host(nxt) == host:
                dist[nxt] = dist[url] + 1
                q.append(nxt)
    return dist


assert get_host("http://a.com/x/y") == "a.com" and get_host("http://a.com") == "a.com"
assert crawl_dfs(start1, HtmlParser(urls1, edges1)) == sorted(want1)
d = crawl_with_dist(start1, HtmlParser(urls1, edges1))
assert d == {start1: 0, "http://news.yahoo.com": 1, "http://news.yahoo.com/news": 1, "http://news.yahoo.com/us": 2}
rng = random.Random(99)
for _ in range(200):
    n = rng.randint(1, 20)
    urls, edges, start = random_case(rng, n, rng.randint(0, 3 * n))
    assert crawl_dfs(start, HtmlParser(urls, edges)) == reference(urls, edges, start)
    assert sorted(crawl_with_dist(start, HtmlParser(urls, edges))) == reference(urls, edges, start)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (BFS crawler lọc theo host):**

```python
class Solution:
    def crawl(self, startUrl: str, htmlParser: 'HtmlParser') -> List[str]:
        host = startUrl.split('/')[2]
        visited = {startUrl}
        q = deque([startUrl])
        while q:
            url = q.popleft()
            for nxt in htmlParser.getUrls(url):
                if nxt not in visited and nxt.split('/')[2] == host:
                    visited.add(nxt)
                    q.append(nxt)
        return list(visited)
```

**Checklist trước khi submit:**

- [ ] Lấy host bằng `split('/')[2]` (hoặc `urlparse`), so sánh bằng `==`, **không** dùng `startswith`.
- [ ] Kiểm tra host **trước khi** đẩy vào hàng đợi (không đi xuyên trang khác host).
- [ ] `visited` khởi tạo có sẵn `startUrl`, đánh dấu **lúc đẩy vào**.
- [ ] Giữ nguyên chuỗi URL (không bỏ `/` cuối, không đổi chữ hoa/thường).
- [ ] Gọi `getUrls` đúng một lần cho mỗi URL; tránh đệ quy sâu nếu không chắc giới hạn.
- [ ] Trả về `list` (thứ tự bất kỳ).